# PTC messages, durable cells, and live CPython state

The harness has four different representations with different jobs:

| Layer | Durable? | Model-visible? | Authority |
|---|---:|---:|---|
| Append-only ledger | Yes | Through bounded views | Historical truth, including incomplete work |
| Notebook | Yes | Selected cells and outputs | Durable PTC transcript/workbench projection |
| CPython namespace | Process-live | Directly available to later cells | Fast working state |
| Model messages | Yes | Yes | Conversation, not execution state |

A variable does not need to be copied into every message. It remains in the live namespace. Safe completed cells can reconstruct it after restart. External effects are never inferred from Python variables or blindly replayed; their ledger receipts decide what completed.

In [ ]:
import hashlib
import json


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


events = [
    {
        "seq": 1,
        "at": "2026-09-01T20:30:00Z",
        "kind": "message.user",
        "status": "completed",
        "text": "Fix the parser and run its tests",
    },
    {
        "seq": 2,
        "at": "2026-09-01T20:30:02Z",
        "kind": "repl.cell",
        "status": "submitted",
        "correlation": "cell-1",
    },
    {
        "seq": 3,
        "at": "2026-09-01T20:30:03Z",
        "kind": "capability.read",
        "status": "started",
        "correlation": "read-1",
        "path": "parser.py",
    },
    {
        "seq": 4,
        "at": "2026-09-01T20:30:04Z",
        "kind": "capability.read",
        "status": "completed",
        "correlation": "read-1",
        "effect": "observed",
    },
    {
        "seq": 5,
        "at": "2026-09-01T20:30:05Z",
        "kind": "repl.cell",
        "status": "completed",
        "correlation": "cell-1",
        "replay_safe": True,
    },
    {
        "seq": 6,
        "at": "2026-09-01T20:30:06Z",
        "kind": "message.assistant",
        "status": "completed",
        "text": "I found the failing parse branch.",
    },
    {
        "seq": 7,
        "at": "2026-09-01T20:30:07Z",
        "kind": "capability.test",
        "status": "started",
        "correlation": "test-1",
        "command": "pytest -q tests/test_parser.py",
    },
]
assert [event["seq"] for event in events] == list(range(1, 8))

## Messages are a view, not the whole session

The model conversation selects only message events. Capability lifecycle, timestamps, failures, and open work remain queryable in the ledger without bloating every prompt.

In [ ]:
messages = [
    {"role": event["kind"].split(".")[1], "content": event["text"]}
    for event in events
    if event["kind"].startswith("message.")
]
print(canonical(messages))
assert len(messages) == 2

## Live state is cheap; durable reconstruction is explicit

The notebook stores code. The worker keeps the resulting objects live. After a crash, only completed cells marked replay-safe are evaluated into a fresh namespace. A brokered write, shell command, or MCP call is represented by a receipt and is not repeated merely because its cell appears in the notebook.

In [ ]:
safe_cells = [
    "target = 'parser.py'",
    "test_command = 'pytest -q tests/test_parser.py'",
    "observations = {'failing_branch': 'parse_union'}",
]
live = {}
for source in safe_cells:
    exec(source, live)

print({name: live[name] for name in ("target", "test_command", "observations")})
assert "target" not in canonical(messages)

restored = {}
for source in safe_cells:
    exec(source, restored)
assert restored["observations"] == live["observations"]

## Reliable on-demand access

A deterministic program reads the ledger at a watermark. Its cache key is `(program, version, task, watermark, arguments)`. The open-execution view below correctly reports the test as unfinished; neither the message transcript nor the live namespace can prove otherwise.

In [ ]:
def compute_view(program, ledger, *, version=1, watermark=None):
    selected = [event for event in ledger if watermark is None or event["seq"] <= watermark]
    if program == "history.model":
        data = {"messages": [event for event in selected if event["kind"].startswith("message.")]}
    elif program == "execution.open":
        open_calls = {}
        for event in selected:
            key = event.get("correlation")
            if key and event["status"] in {"submitted", "started"}:
                open_calls[key] = event["kind"]
            elif key and event["status"] in {"completed", "failed", "blocked", "timeout"}:
                open_calls.pop(key, None)
        data = {"open": open_calls}
    else:
        raise KeyError(program)
    receipt = {
        "program": program,
        "version": version,
        "watermark": selected[-1]["seq"] if selected else 0,
        "evidence_seqs": [event["seq"] for event in selected],
        "data": data,
    }
    receipt["content_hash"] = hashlib.sha256(canonical(receipt).encode()).hexdigest()
    return receipt


open_view = compute_view("execution.open", events)
print(canonical(open_view))
assert open_view["data"] == {"open": {"test-1": "capability.test"}}
assert compute_view("execution.open", events) == open_view